In [48]:
import os
import glob

import pandas as pd
from pathlib import Path
from zoneinfo import ZoneInfo

In [78]:
umass_root = 'data/umass'
umass_original_root = os.path.join(umass_root, 'original') 
umass_homeA = os.path.join(umass_original_root, 'HomeA')
umass_homeA_meters = os.path.join(umass_homeA, 'HomeA_meter')
umass_homeA_weather = os.path.join(umass_homeA, 'HomeA_weather')

years = ['2014', '2015', '2016']

meter2_prefix = 'HomeA-meter2'
meter3_prefix = 'HomeA-meter3'
meter4_prefix = 'HomeA-meter4'
weather_prefix = 'homeA_weather'

LOCAL_TZ="America/New_York"

### 연도별 계량기/날씨 정보 통합

In [81]:
def load_and_resample_hourly(path: Path, suffix: str) -> pd.DataFrame:
    """미터 CSV -> '[kW]' 제거 + 접미사 -> 1시간 평균(kW) 리샘플 -> timeline 컬럼 반환"""
    df = pd.read_csv(path)
    if "Date & Time" not in df.columns:
        raise ValueError(f"'Date & Time' column not found in {path.name}")

    df["Date & Time"] = pd.to_datetime(df["Date & Time"])
    df = df.sort_values("Date & Time").set_index("Date & Time")
    
    # 중복 타임스탬프 있으면 평균 처리(센서 중복 방지)
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()
    if df.index.duplicated().any():
        df = df.groupby(level=0).mean(numeric_only=True)
    
    # 컬럼명 정리: ' [kW]' 제거 + suffix 부여
    rename_map = {c: f"{c.replace(' [kW]', '')}{suffix}" for c in df.columns}
    df = df.rename(columns=rename_map)
    
    # 숫자 컬럼만 대상으로 use 계산
    num_df = df.select_dtypes(include="number").copy()

    use_col = f"use{suffix}" if f"use{suffix}" in num_df.columns else None
    gen_col = f"gen{suffix}" if f"gen{suffix}" in num_df.columns else None

    exclude = set(filter(None, [use_col, gen_col]))
    sum_cols = [c for c in num_df.columns if c not in exclude]

    appliances_sum = num_df[sum_cols].sum(axis=1, min_count=1)
    gen_val = num_df[gen_col] if gen_col in num_df.columns else 0.0
    use_calc = appliances_sum.sub(gen_val, fill_value=0.0)

    df[f"use{suffix}"] = use_calc

    # gen 컬럼 제거
    if gen_col and gen_col in df.columns:
        df = df.drop(columns=[gen_col])

    # 1시간 단위 합(sum)
    df_h = df.resample("1H").sum(numeric_only=True)

    return df_h.reset_index().rename(columns={"Date & Time": "datetime"})

def normalize_cols(cols):
    cleaned = []
    for c in cols:
        c2 = c.strip().lower().replace(" ", "_").replace("-", "_").replace("/", "_")
        c2 = c2.replace("(", "").replace(")", "")
        cleaned.append(c2)
    return cleaned

def load_weather_hourly(path: Path) -> pd.DataFrame:
    """날씨 CSV -> epoch 'time' UTC → America/New_York → naive → 1H 정렬
       수치형은 mean, 범주형은 첫 값 유지(icon), summary는 사용하지 않음(드롭)."""
    w = pd.read_csv(path)
    w.columns = normalize_cols(w.columns)

    # 시간 처리
    if "time" in w.columns:
        w["datetime"] = (
            pd.to_datetime(w["time"], unit="s", utc=True)
              .dt.tz_convert(LOCAL_TZ)
              .dt.tz_localize(None)
        )
        w = w.drop(columns=["time"])
    elif "date_&_time" in w.columns:
        w["datetime"] = pd.to_datetime(w["date_&_time"]); w = w.drop(columns=["date_&_time"])
    elif "datetime" in w.columns:
        w["datetime"] = pd.to_datetime(w["datetime"]); w = w.drop(columns=["datetime"])
    else:
        raise ValueError("날씨 데이터에서 시간 컬럼을 찾지 못했습니다.")

    w = w.sort_values("datetime").set_index("datetime")

    # 수치형은 1H mean
    w_num_h = w.select_dtypes(include="number").resample("1H").mean(numeric_only=True)

    # icon/summary 같은 범주형은 1H 첫 값 사용
    keep_cols = [c for c in w.columns if c not in w_num_h.columns]
    w_cat = w[keep_cols]

    if "icon" in w_cat.columns:
        icon_h = w_cat["icon"].resample("1H").first()
    else:
        icon_h = None

    # summary는 사용하지 않으므로 만들지 않음 (드롭)
    # if "summary" in w_cat.columns:
    #     summary_h = w_cat["summary"].resample("1H").first()

    # 수치 + icon 결합
    w_h = w_num_h.copy()
    if icon_h is not None:
        w_h["icon"] = icon_h

    w_h = w_h.reset_index()  # datetime 복원
    return w_h

In [82]:
for y in years:
    # --- 미터: 1시간 리샘플 후 병합 ---
    m2_h = load_and_resample_hourly(os.path.join(umass_homeA_meters, '{}/{}_{}.csv'.format(y, meter2_prefix, y)), "_m2")
    m3_h = load_and_resample_hourly(os.path.join(umass_homeA_meters, '{}/{}_{}.csv'.format(y, meter3_prefix, y)), "_m3")
    m4_h = load_and_resample_hourly(os.path.join(umass_homeA_meters, '{}/{}_{}.csv'.format(y, meter4_prefix, y)), "_m4")

    df_hourly = (
        m2_h.merge(m3_h, on="datetime", how="outer")
            .merge(m4_h, on="datetime", how="outer")
            .sort_values("datetime")
    )
    
    # --- 날씨 로드 + 병합 (summary는 함수에서 이미 제외, icon은 원본 유지) ---
    weather_h = load_weather_hourly(os.path.join(umass_homeA_weather, '{}_{}.csv'.format(weather_prefix, y)))
    df_with_weather = pd.merge(df_hourly, weather_h, on="datetime", how="left")
    
    if "summary" in df_with_weather.columns:
        df_with_weather = df_with_weather.drop(columns=["summary"])
    
    output_path = os.path.join(umass_root, 'HomeA//HomeA_{}_with_weather.csv'.format(y))
    df_with_weather.to_csv(output_path, index=False)
    
    print(f"저장 완료: {output_path}")

/tmp/ipykernel_1050511/1359103268.py:40: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_h = df.resample("1H").sum(numeric_only=True)
/tmp/ipykernel_1050511/1359103268.py:40: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_h = df.resample("1H").sum(numeric_only=True)
/tmp/ipykernel_1050511/1359103268.py:40: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_h = df.resample("1H").sum(numeric_only=True)
/tmp/ipykernel_1050511/1359103268.py:76: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  w_num_h = w.select_dtypes(include="number").resample("1H").mean(numeric_only=True)
/tmp/ipykernel_1050511/1359103268.py:83: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  icon_h = w_cat["icon"].resample("1H").first()


저장 완료: data/umass/HomeA//HomeA_2014_with_weather.csv


/tmp/ipykernel_1050511/1359103268.py:40: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_h = df.resample("1H").sum(numeric_only=True)
/tmp/ipykernel_1050511/1359103268.py:40: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_h = df.resample("1H").sum(numeric_only=True)
/tmp/ipykernel_1050511/1359103268.py:40: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_h = df.resample("1H").sum(numeric_only=True)
/tmp/ipykernel_1050511/1359103268.py:76: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  w_num_h = w.select_dtypes(include="number").resample("1H").mean(numeric_only=True)
/tmp/ipykernel_1050511/1359103268.py:83: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  icon_h = w_cat["icon"].resample("1H").first()


저장 완료: data/umass/HomeA//HomeA_2015_with_weather.csv


/tmp/ipykernel_1050511/1359103268.py:40: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_h = df.resample("1H").sum(numeric_only=True)
/tmp/ipykernel_1050511/1359103268.py:40: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_h = df.resample("1H").sum(numeric_only=True)
/tmp/ipykernel_1050511/1359103268.py:40: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_h = df.resample("1H").sum(numeric_only=True)
/tmp/ipykernel_1050511/1359103268.py:76: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  w_num_h = w.select_dtypes(include="number").resample("1H").mean(numeric_only=True)
/tmp/ipykernel_1050511/1359103268.py:83: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  icon_h = w_cat["icon"].resample("1H").first()


저장 완료: data/umass/HomeA//HomeA_2016_with_weather.csv


### 최종 통합

In [83]:
HomeA_with_weather_by_yearrs = [
    'data/umass/HomeA/HomeA_2014_with_weather.csv', 
    'data/umass/HomeA/HomeA_2015_with_weather.csv', 
    'data/umass/HomeA/HomeA_2016_with_weather.csv'
]

# 첫 번째 파일 읽기
merged_df = pd.read_csv(HomeA_with_weather_by_yearrs[0])

# 이후 파일 합치기 (첫 행 중복 제거)
for f in HomeA_with_weather_by_yearrs[1:]:
    df = pd.read_csv(f, header=0)
    df = df.iloc[1:]  # 첫 행 제거
    merged_df = pd.concat([merged_df, df], ignore_index=True)

# use_total 컬럼 생성 (use_m2 + use_m3 + use_m4)
merged_df['use_total'] = (
    merged_df[['use_m2', 'use_m3', 'use_m4']].astype(float).sum(axis=1)
)

# 컬럼 순서: use_total을 맨 앞에 배치
cols = ['use_total'] + [c for c in merged_df.columns if c != 'use_total']
merged_df = merged_df[cols]

# 최종 CSV 저장
output_path = "data/umass/HomeA/HomeA_with_weather.csv"
merged_df.to_csv(output_path, index=False, encoding="utf-8")

print(f"{len(HomeA_with_weather_by_yearrs)}개 파일 합쳐서 저장 완료!")
print(f"총 전력량(use_total) 컬럼이 맨 앞에 추가되어 {output_path} 로 저장됨")

3개 파일 합쳐서 저장 완료!
총 전력량(use_total) 컬럼이 맨 앞에 추가되어 data/umass/HomeA/HomeA_with_weather.csv 로 저장됨


모든 필드 더해서 - gen 하면 use. 

2/3/4의 use 다 더하면 HomeA 전력량 

평균 X -> 총량 


----

icon 사용 -> 임베딩


현재 원본이 […] [kW](전력)이라면, 15분 간격 값 4개를 그대로 더하면 단위가 kW의 합이 되고, 우리가 원하는 kWh(전력량) 이 아닙니다. 반드시 간격(Δt)을 곱해 적분해야 해요.

예: 15분 평균이 2 kW가 4개라면

단순합: 2+2+2+2 = 8 kW (단위 오류)

적분: 2×0.25h ×4 = 2 kWh ✅

단, 열이 이미 kWh(전력량)로 기록된 데이터라면 “그냥 합”이 맞습니다. 하지만 이전 코드/데이터는 컬럼명이 […] [kW]였으니 전력(파워) 로 보는 게 타당해요. 게다가 1·15·30·60분이 섞여 있으니 Δt 가중 합이 안전합니다.